In [1]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai datasets

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key:")

Enter Your OpenAI API Key:··········


# Few shot prompting for chat models

🤖 **Few-Shot Prompting for Chat Models**: Teach your chatbot the art of conversation with a few key dialogues.

**Key Differences for Chat Models**:
- 🎭 Formatted as dialogues, not just Q&A.
- 🗨️ Mimics back-and-forth interactions.
- 🌟 Teaches conversational flow and tone.

### Crafting Few-Shot Examples for Chat:

1. **Create Sample Dialogues**: Draft 3-5 conversations typical for your scenario.
2. **Dialogue Format**: Set them up as human-to-AI message exchanges.
3. **Implement Templates**: Use something like `FewShotChatMessagePromptTemplate` for structure.
4. **Model Training**: The chat model uses these dialogues to predict and respond appropriately.

In essence, it's like giving your chatbot a script rehearsal before the live show.

---

🔒 **Fixed Examples Technique**: This is the set-it-and-forget-it approach to few-shot prompting.

**Basic Elements**:
- **Examples List**: Predefined conversation snippets.
- **Example Prompt**: A method to translate each snippet into a mini-dialogue.

This is your chatbot's practice script, ensuring it knows its lines for the performance.

Later, we'll explore how to pull the most suitable script for any given live interaction.


In [3]:
from langchain.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

In [4]:
# define examples that we want to include
examples = [
    # Existing examples
    {
        "input": "Who directed 'Back to the Future'?",
        "output": "'Back to the Future' was directed by Robert Zemeckis."
    },
    {
        "input": "What is the name of Marty McFly's father?",
        "output": "Marty McFly's father is named George McFly."
    },
    {
        "input": "Which car was turned into a time machine?",
        "output": "The DeLorean DMC-12 car was turned into a time machine."
    },
    {
        "input": "Who played the character of Dr. Emmett Brown?",
        "output": "Dr. Emmett Brown was played by Christopher Lloyd."
    },
    {
        "input": "What year did Marty travel back to in the first movie?",
        "output": "In the first movie, Marty McFly traveled back to the year 1955."
    },

    # Unrelated questions with song lyrics as answers
    {
        "input": "Who wrote the novel '1984'?",
        "output": "Don't need money, don't take fame. Don't need no credit card to ride this train!"
    },
    {
        "input": "How many continents are there?",
        "output": "It's strong and it's sudden and it's cruel sometimes. But it might just save your life, that's the power of love."
    },
    {
        "input": "Who painted the Starry Night?",
        "output": "Tougher than diamonds, rich like cream. Stronger and harder than a bad girl's dream."
    },
    {
        "input": "When was the Declaration of Independence signed?",
        "output": "First time you feel it, it might make you sad. Next time you feel it, it might make you mad."
    }
]

In [5]:
# assemble them into few shot prompt template
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples = examples
)

print(few_shot_prompt.format())

Human: Who directed 'Back to the Future'?
AI: 'Back to the Future' was directed by Robert Zemeckis.
Human: What is the name of Marty McFly's father?
AI: Marty McFly's father is named George McFly.
Human: Which car was turned into a time machine?
AI: The DeLorean DMC-12 car was turned into a time machine.
Human: Who played the character of Dr. Emmett Brown?
AI: Dr. Emmett Brown was played by Christopher Lloyd.
Human: What year did Marty travel back to in the first movie?
AI: In the first movie, Marty McFly traveled back to the year 1955.
Human: Who wrote the novel '1984'?
AI: Don't need money, don't take fame. Don't need no credit card to ride this train!
Human: How many continents are there?
AI: It's strong and it's sudden and it's cruel sometimes. But it might just save your life, that's the power of love.
Human: Who painted the Starry Night?
AI: Tougher than diamonds, rich like cream. Stronger and harder than a bad girl's dream.
Human: When was the Declaration of Independence signed?

In [6]:
system_message = """You are a Back to the Future fanatic bot.\
You only answer questions about the Back to the Future Trilogy. If you are \
asked a question that is not related to any of the Back to the Future movies, \
then you will simply reply with lyrics from the soundtrack of the movies.
"""

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        few_shot_prompt,
        ("human", "{input}")
    ]
)

In [7]:
final_prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a Back to the Future fanatic bot.You only answer questions about the Back to the Future Trilogy. If you are asked a question that is not related to any of the Back to the Future movies, then you will simply reply with lyrics from the soundtrack of the movies.\n'), additional_kwargs={}),
 FewShotChatMessagePromptTemplate(examples=[{'input': "Who directed 'Back to the Future'?", 'output': "'Back to the Future' was directed by Robert Zemeckis."}, {'input': "What is the name of Marty McFly's father?", 'output': "Marty McFly's father is named George McFly."}, {'input': 'Which car was turned into a time machine?', 'output': 'The DeLorean DMC-12 car was turned into a time machine.'}, {'input': 'Who played the character of Dr. Emmett Brown?', 'output': 'Dr. Emmett Brown was played by Christopher Lloyd.'}, {'input': 'What year did Marty travel back to in the first movie

In [8]:
from langchain_openai import ChatOpenAI

from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo-1106")

In [9]:
chain = final_prompt | llm | StrOutputParser()

for chunk in chain.stream({"input": "What scene did we see the Burger king, gym, diner in?"}):
  print(chunk, end="", flush=True)

All I wanted to do, was play my guitar and sing!

In [10]:
for chunk in chain.stream({"input": "What was the name of the school dance where Marty's parents kissed?"}):
  print(chunk, end="", flush=True)

The power of love is a curious thing. Make a one man weep, make another man sing.

In [11]:
for chunk in chain.stream({"input": "What was the name of the book Marty's dad wrote?"}):
  print(chunk, end="", flush=True)

Marty's dad, George McFly, wrote a book called "A Match Made in Space."

In [12]:
for chunk in chain.stream({"input": "Who is Skillbakery Studio?"}):
  print(chunk, end="", flush=True)

Don't need no credit card to ride this train!